# Notebook 1: Reading data, selecting rows and columns
*Kaggle Pandas lesson: "Indexing, Selecting & Assigning" (with a 5-minute intro to reading files)*

**The question for this whole module: what makes a stock risky?**

We measure risk as `vol_2025`, the volatility of a stock's daily returns in 2025, and we look for
its drivers: the company's sector (business risk), its cost structure (operating leverage), its debt
(financial leverage), its size, its age, and its beta.

## Learning goals
By the end of this notebook you can
* load a CSV file into a **DataFrame** and describe what a DataFrame, a **Series** and an **index** are;
* pull out data by column name, by **position** (`iloc`) and by **label** (`loc`), and explain the
  one difference between them that bites everybody;
* filter rows with conditions (`==`, `&`, `|`, `isin`, `notnull`);
* add new columns.

## How to use this notebook
Run the cells from top to bottom (Shift+Enter). The **In class** part is worked out for you; read the
short explanations and run each cell. The **Exercises** are yours: replace every `____` with your own
code. Each exercise has a hint (click to expand) and most have a **self-check** cell: run it after your
answer; if it prints "Looks right!" you are done, if it raises an error read the message and try again.

## Setup

`pd.read_csv` reads a comma-separated file into a DataFrame. It accepts a file name or, as here, a URL.
The data are a snapshot of every US-listed company with a market value above $1 billion: figures from
the fiscal-2025 financial statements plus the stock's 2025 return, volatility and beta. All money is in
**millions of dollars**.

In [2]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/assacohen1/fin6040-pandas-data/main/"
firms = pd.read_csv(DATA_URL + "companies_2025.csv")

pd.set_option("display.max_columns", 40)   # the table has 31 columns: show them all
pd.set_option("display.max_rows", 40)      # show up to 40 rows before truncating

| column | meaning |
|---|---|
| `ticker`, `company`, `sector`, `industry_code`, `exchange`, `state` | who the firm is (GICS sector, 11 values) |
| `ipo_date`, `fiscal_year_end` | first trading date (missing for old listings); last day of fiscal 2025 |
| `employees` | thousands of employees |
| `sales`, `cogs`, `sga`, `r_and_d`, `ebitda`, `ebit`, `interest_expense`, `net_income`, `dividends` | income statement items, $ millions |
| `capex`, `cfo` | capital expenditures, cash flow from operations |
| `total_assets`, `total_liabilities`, `total_debt`, `cash`, `book_equity` | balance sheet items, $ millions |
| `shares_out`, `price`, `market_cap` | shares (millions), price at fiscal year end, price x shares ($ millions) |
| `ret_2025` | total stock return in calendar 2025 (0.25 = +25%) |
| `vol_2025` | annualized volatility of daily returns in 2025 (0.25 = 25% per year) |
| `beta_2025` | slope of the stock's daily returns on the S&P 500 (SPY) in 2025 |

Some cells are empty (`NaN`): banks have no cost of goods sold, many firms report no R&D, and so on.
Notebook 4 is about those; until then we mostly use the columns that are complete.

## In class

### 1. What a DataFrame is

In [3]:
firms.head()

,ticker,company,sector,industry_code,exchange,state,ipo_date,fiscal_year_end,employees,sales,cogs,sga,r_and_d,ebitda,ebit,interest_expense,net_income,dividends,capex,cfo,total_assets,total_liabilities,total_debt,cash,book_equity,shares_out,price,market_cap,ret_2025,vol_2025,beta_2025
0,NVDA,Nvidia Corp,Information Technology,453010,NASDAQ,CA,1999-01-22,2026-01-31,42.00,215938.0,55274.0,23076.0,18497.0,137588.0,134745.0,259.0,120067.0,974.0,6042.0,102718.0,206803.0,49510.0,11412.0,62556.0,157293.0,24304.00,191.13,4645223.520,0.3892,0.4962,1.8730
1,GOOGL,Alphabet Inc,Communication Services,502030,NASDAQ,CA,2004-08-19,2025-12-31,190.82,402836.0,141399.0,106405.0,61087.0,155032.0,133896.0,1183.0,132170.0,10315.0,91447.0,164713.0,595281.0,180016.0,66996.0,126843.0,412706.0,12088.00,313.00,3783544.000,0.6600,0.3235,1.0332
2,AAPL,Apple Inc,Information Technology,452020,NASDAQ,CA,1980-12-12,2025-09-30,166.00,416161.0,212960.0,62151.0,34550.0,141050.0,133050.0,NaN,112010.0,15413.0,12715.0,111482.0,359241.0,285508.0,112377.0,54697.0,73733.0,14773.26,254.63,3761715.194,0.0904,0.3251,1.2550
3,MSFT,Microsoft Corp,Information Technology,451030,NASDAQ,WA,1986-03-13,2025-06-30,228.00,281724.0,59831.0,65365.0,32488.0,156528.0,128528.0,2425.0,101832.0,24677.0,64551.0,136162.0,619003.0,275524.0,112184.0,94565.0,343479.0,7434.00,497.41,3697745.940,0.1559,0.2426,0.8762
4,AMZN,Amazon.Com Inc,Consumer Discretionary,255030,NASDAQ,WA,1997-05-15,2025-12-31,1576.00,716924.0,314554.0,273196.0,108521.0,129174.0,86497.0,2274.0,77670.0,0.0,131819.0,139514.0,818042.0,406977.0,178547.0,123329.0,411065.0,10731.00,230.82,2476929.420,0.0521,0.3442,1.3252


`firms` is a **DataFrame**: a table. Each column is a **Series** (one column with a name). The bold
numbers on the left, 0, 1, 2, ..., are the **index**: the row labels. Nobody assigned them; pandas
numbers the rows when it reads the file. The file is sorted by market value, so row 0 is the largest
company in the United States.

You can build a DataFrame yourself from a dictionary of lists, exactly the `m_dict` you built in the
Python class: the keys become column names and each list becomes a column.

In [4]:
pd.DataFrame({"ticker": ["AAPL", "MSFT", "NVDA"],
              "price": [250.0, 500.0, 180.0]})

,ticker,price
0,AAPL,250.0
1,MSFT,500.0
2,NVDA,180.0


A Series is one column on its own. It can carry its own index labels:

In [5]:
pd.Series([0.25, 0.20, 0.45], index=["AAPL", "MSFT", "NVDA"], name="vol")

AAPL    0.25
MSFT    0.20
NVDA    0.45
Name: vol, dtype: float64

`shape` gives (rows, columns); `head(n)` shows the first `n` rows (5 by default).

In [6]:
firms.shape

(2067, 31)

In [7]:
firms.head(3)

,ticker,company,sector,industry_code,exchange,state,ipo_date,fiscal_year_end,employees,sales,cogs,sga,r_and_d,ebitda,ebit,interest_expense,net_income,dividends,capex,cfo,total_assets,total_liabilities,total_debt,cash,book_equity,shares_out,price,market_cap,ret_2025,vol_2025,beta_2025
0,NVDA,Nvidia Corp,Information Technology,453010,NASDAQ,CA,1999-01-22,2026-01-31,42.00,215938.0,55274.0,23076.0,18497.0,137588.0,134745.0,259.0,120067.0,974.0,6042.0,102718.0,206803.0,49510.0,11412.0,62556.0,157293.0,24304.00,191.13,4645223.520,0.3892,0.4962,1.8730
1,GOOGL,Alphabet Inc,Communication Services,502030,NASDAQ,CA,2004-08-19,2025-12-31,190.82,402836.0,141399.0,106405.0,61087.0,155032.0,133896.0,1183.0,132170.0,10315.0,91447.0,164713.0,595281.0,180016.0,66996.0,126843.0,412706.0,12088.00,313.00,3783544.000,0.6600,0.3235,1.0332
2,AAPL,Apple Inc,Information Technology,452020,NASDAQ,CA,1980-12-12,2025-09-30,166.00,416161.0,212960.0,62151.0,34550.0,141050.0,133050.0,NaN,112010.0,15413.0,12715.0,111482.0,359241.0,285508.0,112377.0,54697.0,73733.0,14773.26,254.63,3761715.194,0.0904,0.3251,1.2550


Writing is the mirror image of reading: `to_csv`. `index=False` keeps the 0, 1, 2 ... row numbers out
of the file. (In Colab the file lands on the temporary disk; click the folder icon on the left to see it.)

In [8]:
firms.head(20).to_csv("my_copy.csv", index=False)

### 2. Picking a column
Two ways, same result. The dot form is shorter; the bracket form works for every column name,
including names with spaces or names that clash with a method (`firms.shape` is the attribute, not a column).

In [ ]:
firms.sector

In [ ]:
firms["sector"]

A Series can be indexed with a label (here the row label 0):

In [9]:
firms["sector"][0]

'Information Technology'

In [10]:
type(firms["market_cap"])

pandas.core.series.Series

### 3. `iloc`: selecting by position
`iloc` takes **integer positions**: row selector first, then column selector. A colon means "everything".

In [ ]:
firms.iloc[0]          # the first row, returned as a Series

In [ ]:
firms.iloc[:, 0]       # every row, first column

In [ ]:
firms.iloc[:3, 0]      # rows 0, 1, 2 of the first column

In [ ]:
firms.iloc[1:3, 0]     # rows 1 and 2 (the end of a slice is excluded, as in Python lists)

In [ ]:
firms.iloc[[0, 1, 2], 0]   # a list of positions works too

In [ ]:
firms.iloc[-5:]        # negative positions count from the end: the five smallest firms

### 4. `loc`: selecting by label
`loc` takes **labels**: row labels (the index) and column names. With the default index the row labels
happen to be 0, 1, 2, ... so `loc` and `iloc` look alike, but `loc` is the one that lets you name columns.

In [ ]:
firms.loc[0, "company"]

A Python list of column names picks several columns. We will reuse this list all notebook long:

In [ ]:
SHOW = ["ticker", "sector", "market_cap", "vol_2025", "beta_2025"]
firms.loc[:, SHOW]

**The one thing to remember.** `loc` **includes** the end of a range, `iloc` **excludes** it.
`loc[0:4]` gives five rows, `iloc[0:4]` gives four. To get the first 100 rows write `iloc[:100]` or `loc[:99]`.

In [ ]:
firms.loc[0:4, SHOW]       # rows 0 to 4 inclusive: five rows

In [ ]:
firms.iloc[0:4, :5]        # rows 0 to 3: four rows

`loc` becomes really useful once the index means something. `set_index` makes a column the row label.
It returns a **new** DataFrame; `firms` itself is unchanged (this is true of almost every pandas method).

In [11]:
by_ticker = firms.set_index("ticker")
by_ticker.head(3)

,company,sector,industry_code,exchange,state,ipo_date,fiscal_year_end,employees,sales,cogs,sga,r_and_d,ebitda,ebit,interest_expense,net_income,dividends,capex,cfo,total_assets,total_liabilities,total_debt,cash,book_equity,shares_out,price,market_cap,ret_2025,vol_2025,beta_2025
ticker,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NVDA,Nvidia Corp,Information Technology,453010,NASDAQ,CA,1999-01-22,2026-01-31,42.00,215938.0,55274.0,23076.0,18497.0,137588.0,134745.0,259.0,120067.0,974.0,6042.0,102718.0,206803.0,49510.0,11412.0,62556.0,157293.0,24304.00,191.13,4645223.520,0.3892,0.4962,1.8730
GOOGL,Alphabet Inc,Communication Services,502030,NASDAQ,CA,2004-08-19,2025-12-31,190.82,402836.0,141399.0,106405.0,61087.0,155032.0,133896.0,1183.0,132170.0,10315.0,91447.0,164713.0,595281.0,180016.0,66996.0,126843.0,412706.0,12088.00,313.00,3783544.000,0.6600,0.3235,1.0332
AAPL,Apple Inc,Information Technology,452020,NASDAQ,CA,1980-12-12,2025-09-30,166.00,416161.0,212960.0,62151.0,34550.0,141050.0,133050.0,NaN,112010.0,15413.0,12715.0,111482.0,359241.0,285508.0,112377.0,54697.0,73733.0,14773.26,254.63,3761715.194,0.0904,0.3251,1.2550


In [ ]:
by_ticker.loc["AAPL", SHOW[1:]]     # SHOW[1:] drops "ticker", which is now the index

In [ ]:
by_ticker.loc["AAPL", "market_cap"]

### 5. Selecting rows with conditions
A comparison on a column gives a Series of `True`/`False`, one per row:

In [ ]:
firms.sector == "Utilities"

Put that inside `loc` (or directly inside the brackets) and only the `True` rows survive:

In [ ]:
firms.loc[firms.sector == "Utilities", SHOW]

In [ ]:
len(firms[firms.sector == "Utilities"])    # how many utilities?

Combine conditions with `&` (and) and `|` (or). **Each condition must be in parentheses**, and you must
use `&`/`|`, not the words `and`/`or`, which do not work on Series.

In [ ]:
firms.loc[(firms.sector == "Utilities") & (firms.vol_2025 > 0.30), SHOW]

In [ ]:
firms.loc[(firms.sector == "Information Technology") | (firms.sector == "Health Care"), SHOW]

`isin` is the short way to write a long chain of `|`:

In [ ]:
firms.loc[firms.sector.isin(["Information Technology", "Health Care"]), SHOW]

`notnull()` (and its opposite `isnull()`) selects rows where a value is present (or missing):

In [ ]:
firms.loc[firms.ipo_date.notnull(), ["ticker", "company", "ipo_date"]]

### 6. Adding columns
Assign to a new column name in brackets. A single value is repeated for every row; anything with one
value per row (a list, a `range`, another Series) is used as is. Because the file is sorted by market
value, a running number is a size rank.

In [ ]:
firms["year"] = 2025
firms["size_rank"] = range(1, len(firms) + 1)
firms.loc[:5, ["ticker", "market_cap", "year", "size_rank"]]

Two warnings for later:
* Create columns with brackets, `firms["x"] = ...`. Writing `firms.x = ...` creates an attribute, not a column.
* Assign on the whole DataFrame, not on a filtered piece. `firms[firms.sector == "Utilities"]["x"] = 1`
  looks reasonable, raises a warning, and does nothing. Notebook 4 shows the right way (`loc`).

## Exercises

### Exercise 1: One column

Select the `vol_2025` column and assign it to `vol`. Then check what `type(vol)` returns.

<details><summary>Hint</summary>

`firms.vol_2025` or `firms["vol_2025"]`; a single column is a Series.

</details>

In [ ]:
vol = ____

type(vol)

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert isinstance(vol, pd.Series)
assert vol.shape == (len(firms),)
print("Looks right!")

### Exercise 2: The largest firm

Assign the first record (the largest firm) to `first_row`, and the name of that company (a string) to `first_company`.

<details><summary>Hint</summary>

`iloc[0]` for the row; `loc[0, "company"]` (or `firms.company[0]`) for the name.

</details>

In [ ]:
first_row = ____
first_company = ____

print(first_company)
first_row

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert isinstance(first_row, pd.Series)
assert first_row["ticker"] == 'NVDA'
assert first_company == 'Nvidia Corp'
print("Looks right!")

### Exercise 3: Ten biggest tickers

Create a Series `top10` holding the tickers of the 10 largest firms.

<details><summary>Hint</summary>

`firms.ticker.iloc[:10]`, `firms.loc[:9, "ticker"]` or `firms.ticker.head(10)` all work. Remember that `loc` includes the end label.

</details>

In [ ]:
top10 = ____

top10

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert isinstance(top10, pd.Series)
assert len(top10) == 10
assert top10.iloc[0] == 'NVDA'
assert top10.iloc[9] == 'LLY'
print("Looks right!")

### Exercise 4: Rows by label, columns by name

Create a DataFrame `sample` with the columns `ticker`, `sector`, `market_cap` and `vol_2025` for the rows with index labels 0, 1, 10 and 100.

<details><summary>Hint</summary>

`loc` with a list of row labels and a list of column names.

</details>

In [ ]:
sample = ____

sample

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert sample.shape == (4, 4)
assert list(sample.index) == [0, 1, 10, 100]
assert list(sample.columns) == ["ticker", "sector", "market_cap", "vol_2025"]
print("Looks right!")

### Exercise 5: Loss-makers

Add a column `loss_maker` to `firms` that is `True` when `net_income` is negative. Then create `losers`, the DataFrame of loss-making firms, and print how many there are.

<details><summary>Hint</summary>

A comparison gives the True/False column; a True/False column inside the brackets filters the rows.

</details>

In [ ]:
firms["loss_maker"] = ____
losers = ____

print(len(losers), "firms lost money in fiscal 2025")
losers.loc[:, SHOW]

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert "loss_maker" in firms.columns
assert len(losers) == 454
assert losers.net_income.max() < 0
print("Looks right!")

### Exercise 6: Defensive but wild

Create `defensive_wild`: the firms in the Utilities or Consumer Staples sectors (use `isin`) whose `vol_2025` is at least 0.30, showing the `SHOW` columns only.

<details><summary>Hint</summary>

Two conditions joined with `&`, each in parentheses; `isin` takes a list of sector names.

</details>

In [ ]:
defensive_wild = ____

defensive_wild

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert len(defensive_wild) == 54
assert set(defensive_wild.sector) <= {"Utilities", "Consumer Staples"}
assert defensive_wild.vol_2025.min() >= 0.30
print("Looks right!")

### Exercise 7: Calm versus wild

Create two DataFrames with the `SHOW` columns: `calm`, the firms whose `vol_2025` is at most `LOW`,
and `wild`, the firms whose `vol_2025` is at least `HIGH` (the two thresholds are set in the cell).
Look at both tables, then answer in the cell below: which sectors dominate each table, and which
table has the bigger firms?

<details><summary>Hint</summary>

`firms.loc[condition, SHOW]` twice. `display(x)` shows a table from the middle of a cell; the last line of a cell is shown automatically.

</details>

In [ ]:
LOW = 0.19
HIGH = 1.50

calm = ____
wild = ____

print(len(calm), "calm firms and", len(wild), "wild firms")
display(calm)
wild

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert len(calm) == 36
assert len(wild) == 18
assert calm.vol_2025.max() <= LOW
assert wild.vol_2025.min() >= HIGH
print("Looks right!")

**Your answer:** *Write your answer here (double-click to edit).*

## Finance insight

Before computing a single statistic, the two tables already answer part of our question. The calmest
stocks belong to defensive sectors (utilities, real estate, staples) and to giant profitable firms; the wildest belong to
Information Technology and Health Care (mostly biotech), tend to be smaller, and include many of the
loss-makers from Exercise 5. **Sector and size are visible to the naked eye.** The next notebooks put
numbers on this: how much more volatile, and whether debt or cost structure explains it.